# OpenVLA-OFT T4 Sanity Check

**Goal:** Verify that `moojink/openvla-7b-oft-finetuned-libero-spatial` loads on a free Colab T4 (16 GB VRAM)
and produces actions against the mock LIBERO env.  
**Model:** OpenVLA-OFT (Feb 2025, 97.1% avg on LIBERO, BF16 — no 4-bit quant needed)  
**Expected cost:** $0 (Colab free tier)  
**Expected runtime:** ~10 min

Run all cells top-to-bottom. Cell 7 prints a summary to copy into STATUS.md.

In [ ]:
# 1. Verify we have a GPU
import subprocess
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                        capture_output=True, text=True)
print(result.stdout.strip() or 'No GPU found — switch runtime to T4 GPU')

In [ ]:
# 2. Clone the repo and install deps
import os, sys
if not os.path.exists('/content/vla-bench'):
    !git clone https://github.com/abhid1234/vla-bench.git
%cd /content/vla-bench
!pip install -e '.[real]'
# OpenVLA-OFT's HF modeling code does `from prismatic import ...`. The prismatic
# package lives inside the openvla-oft repo, but pip-installing that repo
# tries to pin torch==2.2.0 and tensorflow==2.15.0 — fights Colab. Instead,
# clone alongside and add to sys.path.
if not os.path.exists('/content/openvla-oft'):
    !git clone https://github.com/moojink/openvla-oft.git /content/openvla-oft
sys.path.insert(0, '/content/openvla-oft')
# Belt-and-suspenders: ensure src is importable even if editable install lags
sys.path.insert(0, '/content/vla-bench/src')

In [ ]:
# 3. Smoke-test the mock harness first (no GPU required)
!python -m vla_bench.cli eval --model mock --env mock-libero --tasks 2 --rollouts 3

In [ ]:
# 4. Check VRAM before loading
import torch
if torch.cuda.is_available():
    free, total = torch.cuda.mem_get_info()
    print(f'VRAM free: {free/1e9:.1f} GB / {total/1e9:.1f} GB total')
else:
    print('CUDA not available — stop here and switch to GPU runtime')

In [ ]:
# 5. Load OpenVLA-OFT (BF16, spatial suite). Expect ~14 GB VRAM, ~3-5 min on T4.
# If VRAM OOM here, the T4 is insufficient — move to Kaggle P100 or Vast.ai 3090.
import time
from vla_bench.models.openvla import OpenVLAOFTModel

t0 = time.time()
model = OpenVLAOFTModel(task_suite='spatial', device='cuda')
print(f'Loaded in {time.time()-t0:.1f}s')

free, total = torch.cuda.mem_get_info()
print(f'VRAM after load: {(total-free)/1e9:.1f} GB used / {total/1e9:.1f} GB total')

In [ ]:
# 6. Single-task sanity run: 1 task, 5 rollouts
from vla_bench.envs.mock import MockLIBEROEnv
from vla_bench.runner import run_task

env = MockLIBEROEnv(max_steps=20)
task_id = 'task_0'

print(f'Task: {env.task_instruction(task_id)}')
metrics = run_task(model, env, task_id, num_rollouts=5, base_seed=42)

print(f'\nSuccess rate : {metrics.success_rate:.1%} ({metrics.successes}/{metrics.rollouts})')
print(f'Mean steps   : {metrics.mean_steps:.1f}')
print(f'Mean inf lat : {metrics.mean_inference_ms:.1f} ms/step')

In [ ]:
# 7. Record findings for STATUS.md
print('=== Copy these numbers to STATUS.md Phase 1 findings ===')
free_after, total = torch.cuda.mem_get_info()
vram_used = (total - free_after) / 1e9
print(f'Model          : OpenVLA-OFT (moojink/openvla-7b-oft-finetuned-libero-spatial)')
print(f'VRAM used      : {vram_used:.1f} GB')
print(f'Fits on T4     : {"YES" if vram_used < 15.5 else "NO — needs Kaggle P100 or Vast.ai 3090"}')
print(f'Inf latency    : {metrics.mean_inference_ms:.1f} ms/step')
print(f'Est per rollout (20 steps): {metrics.mean_inference_ms * 20 / 1000:.1f}s')
est_hrs = metrics.mean_inference_ms * 20 * 20 * 20 / 1000 / 3600  # 20 tasks * 20 rollouts * 20 steps
print(f'Est total run  : {est_hrs:.2f} hrs (20 tasks x 20 rollouts x 20 steps)')
print(f'Free-tier viable: {"YES (Colab)" if est_hrs < 4 else "NO — use Kaggle 30hr/wk quota"}')